# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
# Access the metadata as an object
meta_obj = dataset.metadata
print(f"{meta_obj.name}: {meta_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Print dataset record sets, fields, and their @id
print("Record sets found:")
for rset in dataset.record_sets:
    print(f"- Name: {rset.name} | @id: {rset.id}")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name} | @id: {field.id} | DataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids used in this dataset
record_sets = [rs.id for rs in dataset.record_sets]
print(f"Available record sets: {record_sets}")

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}")
if len(record_sets) > 0:
    example_id = record_sets[0]
    print(f"Columns in {example_id} record set:")
    print(dataframes[example_id].columns.tolist())
    display(dataframes[example_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose the first record set with at least 1 numeric field
import numpy as np

# Find the first record set with numeric fields
numeric_field = None
group_field = None
chosen_set_id = None
for rs in dataset.record_sets:
    df = dataframes.get(rs.id)
    if df is None or df.empty:
        continue
    # Find a numeric field
    num_fields = [f for f in rs.fields if f.data_type in ['Float', 'Number', 'Integer']]
    if num_fields:
        numeric_field = num_fields[0].id  # use @id
        # Optionally find a string/categorical field for grouping
        str_fields = [f for f in rs.fields if f.data_type=='Text']
        group_field = str_fields[0].id if str_fields else None
        chosen_set_id = rs.id
        break

if chosen_set_id is not None and numeric_field in dataframes[chosen_set_id].columns:
    df = dataframes[chosen_set_id]
    # Ensure the numeric field is float
    df_numeric = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df_numeric.quantile(0.90)  # take top 10% for demonstration
    filtered_df = df[df_numeric > threshold].copy()
    print(f"Filtered records from record set '{chosen_set_id}' with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field]].head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (df_numeric[filtered_df.index] - df_numeric[filtered_df.index].mean()) / df_numeric[filtered_df.index].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field}: (showing top 5)")
        print(grouped_df.head())
else:
    print("No suitable numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_set_id is not None and numeric_field in dataframes[chosen_set_id].columns:
    df = dataframes[chosen_set_id]
    # Drop NA for plotting
    vals = pd.to_numeric(df[numeric_field], errors='coerce').dropna()
    plt.figure(figsize=(7,4))
    sns.histplot(vals, bins=30, kde=True)
    plt.xlabel(numeric_field)
    plt.title(f'Distribution of {numeric_field} in record set {chosen_set_id}')
    plt.show()
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f'{numeric_field} by {group_field} in {chosen_set_id}')
        plt.show()
else:
    print("No visualization available (no suitable numeric field found).")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library via its Croissant schema. We reviewed record set structures and field `@id`s, extracted records as DataFrames, and performed basic exploratory analysis and visualizations based on the available numeric fields.

- The dataset includes experimental results regarding protein abundance, modifications, and sequences from human mast cells.
- Data was processed to demonstrate field normalization and group-wise aggregation, highlighting the utility of field `@id`s for robust referencing.
- Visualizations of value distributions offer insights into mass spectrometry results, which can guide further downstream biological interpretation or model development.

For further analysis, leverage additional fields, apply advanced statistics, and consider domain-specific questions relevant to biological interpretation or hypothesis testing.